# Datengenerierung und Zugriff
Hier möchte ich Funktionen für verschiedene Parameter kombinieren. Außerdem möchte ich die Generierung der Testdaten von der Abfrage trennen damit die Generierung später einfach durch eine Datenbankabfrage ersetzt werden kann und die Abfrage kürzer und übersichtlicher wird.

In [ ]:
import pandas as pd
import numpy as np
from typing import Literal

## Generate

In [ ]:
Direction = Literal["consumption", "generation"]
Frequency = Literal["15min", "1h"]

TZ = "Europe/Vienna"

def create_data(
    meter_id: str,
    direction: Direction,
    start_date: pd.Timestamp,
    end_date: pd.Timestamp,
    freq: Frequency,
    forecast: bool
) -> pd.DataFrame:
    timestamps = pd.date_range(start=start_date, end=end_date, freq=freq, inclusive="left")
    
    base = np.sin(np.linspace(0, 2*np.pi, len(timestamps))) + 1

    rng = np.random.default_rng(42)
    noise = rng.normal(0, 0.1, len(timestamps))
    
    values = (base + noise).clip(min=0)
    
    df = pd.DataFrame({
        "timestamp": timestamps,
        "meter_id": meter_id,
        "forecast": forecast,
        "direction": direction,
        "value": values
    })

    return df.astype({
        "meter_id": "string",
        "forecast": "bool",
        "direction": "string",
        "value": "float64"
    })

def create_actual_and_forecast(
    meter_id: str,
    direction: Direction,
    start_date: pd.Timestamp
) -> pd.DataFrame:
    now = pd.Timestamp.now(tz=TZ)
    today_00 = now.normalize()
    yesterday_00 = today_00 - pd.Timedelta(days=1)
    day_after_tomorrow_00 = today_00 + pd.Timedelta(days=2)

    df_actual = create_data(
        meter_id=meter_id,
        direction=direction,
        start_date=start_date,
        end_date=yesterday_00,
        freq="15min",
        forecast=False
    )

    df_forecast = create_data(
        meter_id=meter_id,
        direction=direction,
        start_date=yesterday_00,
        end_date=day_after_tomorrow_00,
        freq="1h",
        forecast=True
    )

    return pd.concat([df_actual, df_forecast])

def create_testdata() -> pd.DataFrame:
    # start_date = pd.Timestamp("2026-04-10", tz=TZ).normalize()
    start_date = pd.Timestamp.now(tz=TZ) - pd.Timedelta(days=3)

    mp_1 = create_actual_and_forecast("mp_1", "consumption", start_date)
    mp_2 = create_actual_and_forecast("mp_2", "generation", start_date)

    return pd.concat([mp_1, mp_2])

df = create_testdata()

print(df.to_string())

                           timestamp meter_id  forecast    direction     value
0   2026-04-10 18:09:29.142444+02:00     mp_1     False  consumption  1.030472
1   2026-04-10 18:24:29.142444+02:00     mp_1     False  consumption  0.948777
2   2026-04-10 18:39:29.142444+02:00     mp_1     False  consumption  1.180449
3   2026-04-10 18:54:29.142444+02:00     mp_1     False  consumption  1.251795
4   2026-04-10 19:09:29.142444+02:00     mp_1     False  consumption  1.014529
5   2026-04-10 19:24:29.142444+02:00     mp_1     False  consumption  1.130725
6   2026-04-10 19:39:29.142444+02:00     mp_1     False  consumption  1.324311
7   2026-04-10 19:54:29.142444+02:00     mp_1     False  consumption  1.329617
8   2026-04-10 20:09:29.142444+02:00     mp_1     False  consumption  1.408270
9   2026-04-10 20:24:29.142444+02:00     mp_1     False  consumption  1.372211
10  2026-04-10 20:39:29.142444+02:00     mp_1     False  consumption  1.591745
11  2026-04-10 20:54:29.142444+02:00     mp_1     Fa

## Access

In [ ]:
get

## Json

In [ ]:
tools = [
  {
    "type": "function",
    "function": {
      "name": "generate_series",
      "description": "Generate synthetic electricity demand (consumption) time series data.",
      "parameters": {
        "type": "object",
        "properties": {
          "start": {
            "type": "string",
            "format": "date-time",
            "description": "Start timestamp in ISO 8601 format (e.g. 2026-01-01T00:00:00)"
          },
          "end": {
            "type": "string",
            "format": "date-time",
            "description": "End timestamp in ISO 8601 format (e.g. 2026-01-01T23:00:00)"
          },
          "pattern": {
            "type": "string",
            "enum": ["residential", "industrial"],
            "description": "Consumption pattern type. Residential has peaks in the morning and evening, industrial peaks during working hours.",
            "default": "residential"
          }
        },
        "required": ["start", "end"]
      }
    }
  },
  {
    "type": "function",
    "function": {
      "name": "generate_production_series",
      "description": "Generate synthetic electricity supply (production) time series data.",
      "parameters": {
        "type": "object",
        "properties": {
          "start": {
            "type": "string",
            "format": "date-time",
            "description": "Start timestamp in ISO 8601 format (e.g. 2026-01-01T00:00:00)"
          },
          "end": {
            "type": "string",
            "format": "date-time",
            "description": "End timestamp in ISO 8601 format (e.g. 2026-01-01T23:00:00)"
          },
          "pattern": {
            "type": "string",
            "enum": ["residential", "industrial"],
            "description": "Production pattern type. Residential may simulate small-scale generation (e.g. solar), industrial simulates large-scale continuous production.",
            "default": "residential"
          }
        },
        "required": ["start", "end"]
      }
    }
  }
]

In [ ]:
def get_timeseries(
    meter_id: str,
    metric: str,
    start_time: str,
    end_time: str,
    include_forecast: bool = True
):
    pass